## Structured output

Models -> requesst to provide in format matching the schema
useful for parsing

### Pydantic

Richest feature set with field validation, desc and nested structures

In [13]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(model="llama3.2", model_provider="ollama")
model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2')

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="Year of movie release")
    director: str= Field(description="Director of the movie")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, model='llama3.2'), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'Year of movie release', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'title': 'Director', 'type': 'string'}}, 'required': ['title', 'year', 'director'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [4]:
res = model_with_structure.invoke("Provide details about Matrix")
res

Movie(title='The Matrix', year=1999, director='The Wachowskis,')

In [5]:
class Movie(BaseModel):
    title: str=Field(..., description="The title of the movie")
    year: int=Field(..., description="Year of movie release")
    director: str= Field(..., description="Director of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure

res = model_with_structure.invoke("Provide details about Matrix")

In [6]:
res

{'raw': AIMessage(content='{"title": "The Matrix", "year": 1999, "director": "The Wachowskis" }', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-07-18T10:45:59.530731Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1408103042, 'load_duration': 403561125, 'prompt_eval_count': 29, 'prompt_eval_duration': 122007000, 'eval_count': 27, 'eval_duration': 801589000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019f74d5-3668-7b21-93f0-c0194deaa01d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 27, 'total_tokens': 56}),
 'parsed': Movie(title='The Matrix', year=1999, director='The Wachowskis'),
 'parsing_error': None}

In [12]:
print(model.profile)

None


### DataClasses

Class containing mainly data, but no restrictions.
create using @dataclass decorator

In [18]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name:str = Field(description="Name of the person")
    email:str = Field(description="email of the person")
    phone:str = Field(description="phone of the person")

agent = create_agent(model=model, response_format=ContactInfo)

res = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from John Doe, john@doe.com, 9876543212"}]
})

res

{'messages': [HumanMessage(content='Extract contact info from John Doe, john@doe.com, 9876543212', additional_kwargs={}, response_metadata={}, id='f350adbb-62b1-43ae-9ff0-5e3ea907684c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-07-18T12:59:51.743869Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22187173834, 'load_duration': 19878022709, 'prompt_eval_count': 197, 'prompt_eval_duration': 930556000, 'eval_count': 37, 'eval_duration': 1263781000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019f754f-757c-7713-9bed-032ab035010a-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@doe.com', 'phone': '9876543212', 'name': 'John Doe'}, 'id': '912a0beb-9e84-4b65-9420-508dcd003087', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_tokens': 37, 'total_tokens': 234}),
  ToolMessage(content="Returning structured response: name='Joh

In [19]:
res["structured_response"]

ContactInfo(name='John Doe', email='john@doe.com', phone='9876543212')

In [20]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name:str
    email:str
    phone:str

agent = create_agent(model=model, response_format=ContactInfo)

res = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from John Doe, john@doe.com, 9876543212"}]
})

res["structured_response"]

ContactInfo(name='John Doe', email='john@doe.com', phone='9876543212')